# LiteRT-LM vendor-matrix T4 row on Colab

**First: Runtime → Change runtime type → T4 GPU.**

Runs the stock `www/timing.html` bare flow (the same 16-chunk decode that produced the Metal row in `out/vendor-matrix.json`) through headless Chrome on a T4 and emits one vendor-matrix row: adapter, browser, shader-f16/timestamp-query presence, steady tok/s, and the pinned model SHA. Cell 1 installs the NVIDIA Vulkan userspace Chrome needs (zero-tvm's recipe, after Chrome's colab-headless guide); if it prints `llvmpipe` instead of `Tesla T4`, stop and report the driver version from `nvidia-smi`.

Nothing is uploaded anywhere. Copy-back instructions are in the last cell. Method, harness and records: https://github.com/abgnydn/litert-capture. Independent work, not affiliated with Google.

In [ ]:
# 1) Node 22 + the NVIDIA Vulkan userspace matching the driver
import subprocess
drv = subprocess.check_output('nvidia-smi --query-gpu=driver_version --format=csv,noheader', shell=True).decode().split('.')[0].strip()
print('NVIDIA driver major:', drv, '-> installing libnvidia-gl-' + drv)
!curl -fsSL https://deb.nodesource.com/setup_22.x | bash - > /dev/null 2>&1
!apt-get -qq update > /dev/null
# libasound2 is libasound2t64 on Colab's Ubuntu 24.04; one missing package name aborts the whole install
!apt-get -qq install -y nodejs vulkan-tools libnvidia-gl-{drv} libnss3 libatk1.0-0 libatk-bridge2.0-0 libcups2 libgbm1 libasound2t64 libxcomposite1 libxdamage1 libxrandr2 libxkbcommon0 libxfixes3 libdrm2 libpango-1.0-0 libcairo2 libatspi2.0-0 fonts-liberation > /dev/null
!rm -f /usr/share/vulkan/icd.d/lvp_icd*.json
print('--- Vulkan device (want Tesla T4, not llvmpipe): ---')
!vulkaninfo --summary 2>/dev/null | grep -E 'deviceName|driverName' || echo 'NO VULKAN DEVICE'

In [ ]:
# 2) Unpack the timing page + driver, install puppeteer, fetch the pinned model (2 GB, SHA-checked)
import base64, os, hashlib
files = {"timing.html":"PCFkb2N0eXBlIGh0bWw+CjxtZXRhIGNoYXJzZXQ9InV0Zi04Ij4KPHRpdGxlPkxpdGVSVC1MTSB3ZWIgZGVjb2RlIHRpbWluZzwvdGl0bGU+CjxwcmUgaWQ9ImxvZyIgc3R5bGU9IndoaXRlLXNwYWNlOiBwcmUtd3JhcDsgZm9udDogMTJweCBtb25vc3BhY2UiPjwvcHJlPgoKPCEtLSBXaGVyZSBkb2VzIGEgZGVjb2RlZCB0b2tlbidzIHRpbWUgZ28/CiAgICAgbW9kZT1iYXJlICAgOiBvbmx5IG1hcEFzeW5jIGlzIHdyYXBwZWQgKHR3byB0aW1lc3RhbXBzIHBlciB0b2tlbiksIHNvIHRoZQogICAgICAgICAgICAgICAgICAgbWVhc3VyZWQgcmF0ZSBpcyBhcyBjbG9zZSB0byB1bmhvb2tlZCBhcyB0aGlzIG1ldGhvZCBhbGxvd3MuCiAgICAgICAgICAgICAgICAgICBObyBHUFUgdGltZXN0YW1wczsgQ1BVLXNpZGUgdnMgd2FpdGluZy1vbi1HUFUgc3BsaXQgb25seS4KICAgICBtb2RlPWdwdSAgICA6IGVuYWJsZXMgdGltZXN0YW1wLXF1ZXJ5IG9uIHRoZSBkZXZpY2UgTGl0ZVJULUxNIHJlcXVlc3RzIGFuZAogICAgICAgICAgICAgICAgICAgaW5qZWN0cyB0aW1lc3RhbXBXcml0ZXMgaW50byBpdHMgY29tcHV0ZSBwYXNzZXM6IEdQVQogICAgICAgICAgICAgICAgICAgZXhlY3V0aW9uIHRpbWUgcGVyIHBhc3MsIHJlc29sdmVkIHZpYSByZXNvbHZlUXVlcnlTZXQgKwogICAgICAgICAgICAgICAgICAgY29weUJ1ZmZlclRvQnVmZmVyLiBXaGVuIHRoZSBhZGFwdGVyIGxhY2tzIHRpbWVzdGFtcC1xdWVyeQogICAgICAgICAgICAgICAgICAgZ3B1Tm90ZSByZWNvcmRzICd0aW1lc3RhbXAtcXVlcnkgdW5hdmFpbGFibGUnLgogICAgIG1vZGU9a2VybmVsIDogZm9yIGEgZmV3IHRva2VucywgYWRkaXRpb25hbGx5IHNwbGl0cyBldmVyeSBkaXNwYXRjaCBpbnRvCiAgICAgICAgICAgICAgICAgICBpdHMgb3duIHRpbWVzdGFtcGVkIHBhc3MgKHJlcGxheWluZyBwaXBlbGluZSBhbmQgYmluZAogICAgICAgICAgICAgICAgICAgZ3JvdXBzKTogR1BVIGV4ZWN1dGlvbiB0aW1lIHBlciBrZXJuZWwuIEhlYXZpbHkgcGVydHVyYnMKICAgICAgICAgICAgICAgICAgIHRoZSBydW4sIHNvIG9ubHkgdGhlIHBlci1rZXJuZWwgYXR0cmlidXRpb24gaXMgbWVhbmluZ2Z1bC4KPCAgICAgQ1BVLXNpZGUgbnVtYmVycyBjb21lIGZyb20gbW9kZT1iYXJlIG9ubHkuIE9wdGlvbmFsID92PSAvID9jaHVua3M9CiAgICAgdHVuZSB0aGUgYnVuZGxlIGFuZCB0b2tlbiBjb3VudDsgZGVmYXVsdCBpcyB2IDAuMTcuMSwgMjAwIGNodW5rcy4KICAgICBST0FETUFQIDE3OiBjcHUtYmF0Y2ggZmFsbGJhY2sgbGl2ZXMgaW4gd3d3L2tlcm5lbC0wMTEyLmh0bWwgKGJhdGNoZXMgb2YKICAgICA0OCBHUFUtdGltZXN0YW1wZWQgcnVucyB2cyA0ODAgSmF2YVNjcmlwdC10aW1lZCBydW5zLCByZXN1bHRzIHRpbWluZwogICAgIGdwdS10aW1lc3RhbXBzIHZzIGNwdS1iYXRjaCk7IHRoaXMgcGFnZSBpbnN0ZWFkIHJlY29yZHMgZ3B1Tm90ZSB3aGVuCiAgICAgdGltZXN0YW1wLXF1ZXJ5IGlzIHVuYXZhaWxhYmxlLiBEb2N1bWVudGVkIGZyb20gZXhpc3RpbmcgcHJhY3RpY2U7CiAgICAgZGVmYXVsdCB1bmNoYW5nZWQuIC0tPgo8c2NyaXB0PgooKCkgPT4gewogIGNvbnN0IHBhcmFtcyA9IG5ldyBVUkxTZWFyY2hQYXJhbXMobG9jYXRpb24uc2VhcmNoKQogIGNvbnN0IE1PREUgPSBwYXJhbXMuZ2V0KCdtb2RlJykgPz8gJ2JhcmUnCiAgY29uc3QgVCA9ICh3aW5kb3cuX190aW1pbmcgPSB7CiAgICBtb2RlOiBNT0RFLCBtYXJrczogW10sIG1hcHM6IFtdLCBwYXNzZXM6IFtdLCBwaXBlbGluZXM6IFtdLCBncHVFcnJvcnM6IFtdLAogICAgY2h1bms6IDAsIGFybWVkOiBmYWxzZSwgZ3B1Tm90ZTogJycsIGRlYnVnR3JvdXBDYWxsczogMCwKICB9KQogIHdpbmRvdy5fX21hcmsgPSAobmFtZSkgPT4gVC5tYXJrcy5wdXNoKHsgbmFtZSwgdDogcGVyZm9ybWFuY2Uubm93KCkgfSkKCiAgY29uc3Qgb3JpZ01hcEFzeW5jID0gR1BVQnVmZmVyLnByb3RvdHlwZS5tYXBBc3luYwogIEdQVUJ1ZmZlci5wcm90b3R5cGUubWFwQXN5bmMgPSBmdW5jdGlvbiAoLi4uYXJncykgewogICAgY29uc3QgcmVjID0geyBjYWxsOiBwZXJmb3JtYW5jZS5ub3coKSwgcmVzb2x2ZTogMCwgY2h1bms6IFQuY2h1bmsgfQogICAgVC5tYXBzLnB1c2gocmVjKQogICAgY29uc3QgcCA9IG9yaWdNYXBBc3luYy5hcHBseSh0aGlzLCBhcmdzKQogICAgcC50aGVuKCgpID0+IHsgcmVjLnJlc29sdmUgPSBwZXJmb3JtYW5jZS5ub3coKSB9LCAoKSA9PiB7IHJlYy5yZXNvbHZlID0gLTEgfSkKICAgIHJldHVybiBwCiAgfQoKICBpZiAoTU9ERSA9PT0gJ2JhcmUnKSByZXR1cm4KCiAgY29uc3QgU1BMSVQgPSBNT0RFID09PSAna2VybmVsJwogIGNvbnN0IE1BWFEgPSA0MDk2CiAgY29uc3QgTUFYX1NFVFMgPSBTUExJVCA/IDEwIDogMQogIGxldCBkZXZpY2UgPSBudWxsCiAgY29uc3Qgc2V0cyA9IFtdCiAgbGV0IG5leHRRdWVyeSA9IDAKICBjb25zdCBhbGxvY1F1ZXJ5ID0gKCkgPT4gewogICAgY29uc3Qgc2kgPSBNYXRoLmZsb29yKG5leHRRdWVyeSAvIE1BWFEpCiAgICBpZiAoIWRldmljZSB8fCBzaSA+PSBNQVhfU0VUUykgcmV0dXJuIG51bGwKICAgIGlmICghc2V0c1tzaV0pIHNldHNbc2ldID0gZGV2aWNlLmNyZWF0ZVF1ZXJ5U2V0KHsgdHlwZTogJ3RpbWVzdGFtcCcsIGNvdW50OiBNQVhRIH0pCiAgICBjb25zdCBxID0gbmV4dFF1ZXJ5ICUgTUFYUQogICAgbmV4dFF1ZXJ5ICs9IDIKICAgIHJldHVybiB7IHNpLCBxIH0KICB9CgogIGNvbnN0IG9yaWdSZXF1ZXN0RGV2aWNlID0gR1BVQWRhcHRlci5wcm90b3R5cGUucmVxdWVzdERldmljZQogIEdQVUFkYXB0ZXIucHJvdG90eXBlLnJlcXVlc3REZXZpY2UgPSBhc3luYyBmdW5jdGlvbiAoZGVzYyA9IHt9KSB7CiAgICBjb25zdCBmZWF0cyA9IG5ldyBTZXQoZGVzYy5yZXF1aXJlZEZlYXR1cmVzID8/IFtdKQogICAgaWYgKHRoaXMuZmVhdHVyZXMuaGFzKCd0aW1lc3RhbXAtcXVlcnknKSkgZmVhdHMuYWRkKCd0aW1lc3RhbXAtcXVlcnknKQogICAgY29uc3QgZGV2ID0gYXdhaXQgb3JpZ1JlcXVlc3REZXZpY2UuY2FsbCh0aGlzLCB7IC4uLmRlc2MsIHJlcXVpcmVkRmVhdHVyZXM6IFsuLi5mZWF0c10gfSkKICAgIGlmICghZGV2aWNlICYmIGRldi5mZWF0dXJlcy5oYXMoJ3RpbWVzdGFtcC1xdWVyeScpKSB7CiAgICAgIGRldmljZSA9IGRldgogICAgICAvLyBTcGxpdHRpbmcgcGFzc2VzIG11c3Qgbm90IGdvIHVubm90aWNlZCBpZiBpdCBwcm9kdWNlcyBpbnZhbGlkIGNvbW1hbmRzLgogICAgICBkZXYuYWRkRXZlbnRMaXN0ZW5lcigndW5jYXB0dXJlZGVycm9yJywgKGUpID0+IHsKICAgICAgICBpZiAoVC5ncHVFcnJvcnMubGVuZ3RoIDwgMjApIFQuZ3B1RXJyb3JzLnB1c2goU3RyaW5nKGUuZXJyb3I/Lm1lc3NhZ2UgPz8gZSkuc2xpY2UoMCwgMzAwKSkKICAgICAgfSkKICAgIH0KICAgIHJldHVybiBkZXYKICB9CgogIC8vIFBpcGVsaW5lIGlkcyBpbiBjcmVhdGlvbiBvcmRlciwgd2l0aCB0aGUgc2hhZGVyJ3MgU0hBLTI1NiBhbmQgc2l6ZSBzbyB0aGUKICAvLyBkcml2ZXIgY2FuIGNoZWNrIHRoZXkgbGluZSB1cCB3aXRoIHRoZSBjYXB0dXJlIHJ1bidzIG1hbmlmZXN0LiBUaGUgaGFzaCBpcwogIC8vIHRoZSBpZGVudGl0eTsgdGhlIHNpemUgaXMga2VwdCBiZWNhdXNlIHRoZSBjb21taXR0ZWQgcmVjb3JkcyBqb2luIG9uIGl0LgogIC8vIGRpZ2VzdCgpIGlzIGFzeW5jIGFuZCBjcmVhdGVTaGFkZXJNb2R1bGUgaXMgbm90LCBzbyB0aGUgaGFzaGVzIGFyZSBmaWxsZWQKICAvLyBpbnRvIHRoZSBwaXBlbGluZSByZWNvcmRzIGFmdGVyd2FyZHMsIGJlZm9yZSB0aGUgcGFnZSBkZWNsYXJlcyBpdHNlbGYgZG9uZS4KICAvLyBJZiBjcnlwdG8uc3VidGxlIGlzIG1pc3NpbmcgdGhlIHJlY29yZCBzYXlzIHNvIHJhdGhlciB0aGFuIGNhcnJ5aW5nIGEKICAvLyBtYWRlLXVwIGhhc2guCiAgY29uc3Qgc2hhZGVySW5mbyA9IG5ldyBXZWFrTWFwKCkKICBjb25zdCBwaXBlbGluZUlkcyA9IG5ldyBXZWFrTWFwKCkKICBjb25zdCBwaXBlbGluZVNoYWRlciA9IFtdIC8vIHBpcGVsaW5lIGlkIC0+IGl0cyBzaGFkZXIncyBpbmZvIHJlY29yZAogIGNvbnN0IGhhc2hKb2JzID0gW10KICBjb25zdCBvcmlnQ3JlYXRlU2hhZGVyID0gR1BVRGV2aWNlLnByb3RvdHlwZS5jcmVhdGVTaGFkZXJNb2R1bGUKICBHUFVEZXZpY2UucHJvdG90eXBlLmNyZWF0ZVNoYWRlck1vZHVsZSA9IGZ1bmN0aW9uICguLi5hKSB7CiAgICBjb25zdCBtID0gb3JpZ0NyZWF0ZVNoYWRlci5hcHBseSh0aGlzLCBhKQogICAgY29uc3QgY29kZSA9IGFbMF0/LmNvZGUgPz8gJycKICAgIGNvbnN0IGluZm8gPSB7IGJ5dGVzOiBhWzBdPy5jb2RlPy5sZW5ndGggPz8gLTEsIHNoYTI1NjogbnVsbCB9CiAgICBzaGFkZXJJbmZvLnNldChtLCBpbmZvKQogICAgaWYgKGdsb2JhbFRoaXMuY3J5cHRvPy5zdWJ0bGUpIHsKICAgICAgaGFzaEpvYnMucHVzaChjcnlwdG8uc3VidGxlLmRpZ2VzdCgnU0hBLTI1NicsIG5ldyBUZXh0RW5jb2RlcigpLmVuY29kZShjb2RlKSkudGhlbigoZCkgPT4gewogICAgICAgIGluZm8uc2hhMjU2ID0gWy4uLm5ldyBVaW50OEFycmF5KGQpXS5tYXAoKGIpID0+IGIudG9TdHJpbmcoMTYpLnBhZFN0YXJ0KDIsICcwJykpLmpvaW4oJycpCiAgICAgIH0pKQogICAgfSBlbHNlIFQuc2hhZGVyU2hhMjU2VW5hdmFpbGFibGUgPSB0cnVlCiAgICByZXR1cm4gbQogIH0KICBjb25zdCBub3RlUGlwZWxpbmUgPSAocCwgZGVzYykgPT4gewogICAgY29uc3QgaW5mbyA9IHNoYWRlckluZm8uZ2V0KGRlc2M/LmNvbXB1dGU/Lm1vZHVsZSkgPz8geyBieXRlczogLTEsIHNoYTI1NjogbnVsbCB9CiAgICBwaXBlbGluZUlkcy5zZXQocCwgVC5waXBlbGluZXMubGVuZ3RoKQogICAgcGlwZWxpbmVTaGFkZXIucHVzaChpbmZvKQogICAgVC5waXBlbGluZXMucHVzaCh7IGlkOiBULnBpcGVsaW5lcy5sZW5ndGgsIHNoYWRlckJ5dGVzOiBpbmZvLmJ5dGVzLCBzaGFkZXJTaGEyNTY6IG51bGwgfSkKICB9CiAgd2luZG93Ll9fc2V0dGxlSGFzaGVzID0gYXN5bmMgKCkgPT4gewogICAgYXdhaXQgUHJvbWlzZS5hbGwoaGFzaEpvYnMpCiAgICBULnBpcGVsaW5lcy5mb3JFYWNoKChwLCBpKSA9PiB7IHAuc2hhZGVyU2hhMjU2ID0gcGlwZWxpbmVTaGFkZXJbaV0uc2hhMjU2IH0pCiAgfQogIGNvbnN0IG9yaWdDcmVhdGVQaXBlID0gR1BVRGV2aWNlLnByb3RvdHlwZS5jcmVhdGVDb21wdXRlUGlwZWxpbmUKICBHUFVEZXZpY2UucHJvdG90eXBlLmNyZWF0ZUNvbXB1dGVQaXBlbGluZSA9IGZ1bmN0aW9uICguLi5hKSB7CiAgICBjb25zdCBwID0gb3JpZ0NyZWF0ZVBpcGUuYXBwbHkodGhpcywgYSkKICAgIG5vdGVQaXBlbGluZShwLCBhWzBdKQogICAgcmV0dXJuIHAKICB9CiAgY29uc3Qgb3JpZ0NyZWF0ZVBpcGVBc3luYyA9IEdQVURldmljZS5wcm90b3R5cGUuY3JlYXRlQ29tcHV0ZVBpcGVsaW5lQXN5bmMKICBHUFVEZXZpY2UucHJvdG90eXBlLmNyZWF0ZUNvbXB1dGVQaXBlbGluZUFzeW5jID0gZnVuY3Rpb24gKC4uLmEpIHsKICAgIHJldHVybiBvcmlnQ3JlYXRlUGlwZUFzeW5jLmFwcGx5KHRoaXMsIGEpLnRoZW4oKHApID0+IHsgbm90ZVBpcGVsaW5lKHAsIGFbMF0pOyByZXR1cm4gcCB9KQogIH0KCiAgY29uc3QgZW5jRGV2aWNlID0gbmV3IFdlYWtNYXAoKQogIGNvbnN0IG9yaWdDcmVhdGVFbmMgPSBHUFVEZXZpY2UucHJvdG90eXBlLmNyZWF0ZUNvbW1hbmRFbmNvZGVyCiAgR1BVRGV2aWNlLnByb3RvdHlwZS5jcmVhdGVDb21tYW5kRW5jb2RlciA9IGZ1bmN0aW9uICguLi5hKSB7CiAgICBjb25zdCBlID0gb3JpZ0NyZWF0ZUVuYy5hcHBseSh0aGlzLCBhKQogICAgZW5jRGV2aWNlLnNldChlLCB0aGlzKQogICAgcmV0dXJuIGUKICB9CgogIC8vIHN0YXRlIHBlciBwYXNzIG9iamVjdCB0aGUgY2FsbGVyIGhvbGRzLiBgY3VyYCBpcyB0aGUgcmVhbCBwYXNzIGNvbW1hbmRzCiAgLy8gY3VycmVudGx5IGdvIHRvOyBpbiBzcGxpdCBtb2RlIGl0IGNoYW5nZXMgYXQgZXZlcnkgZGlzcGF0Y2guCiAgY29uc3Qgc3QgPSBuZXcgV2Vha01hcCgpCiAgY29uc3Qgd2l0aFRzID0gKGRlc2MsIGEpID0+ICh7CiAgICAuLi4oZGVzYyA/PyB7fSksCiAgICB0aW1lc3RhbXBXcml0ZXM6IHsgcXVlcnlTZXQ6IHNldHNbYS5zaV0sIGJlZ2lubmluZ09mUGFzc1dyaXRlSW5kZXg6IGEucSwgZW5kT2ZQYXNzV3JpdGVJbmRleDogYS5xICsgMSB9LAogIH0pCiAgY29uc3QgbmV3UmVjID0gKGEsIGNodW5rKSA9PiB7CiAgICBjb25zdCByZWMgPSB7IHNpOiBhLnNpLCBxOiBhLnEsIGNodW5rLCBkaXNwYXRjaGVzOiAwLCBwaWQ6IC0xIH0KICAgIFQucGFzc2VzLnB1c2gocmVjKQogICAgcmV0dXJuIHJlYwogIH0KCiAgY29uc3Qgb3JpZ0JlZ2luID0gR1BVQ29tbWFuZEVuY29kZXIucHJvdG90eXBlLmJlZ2luQ29tcHV0ZVBhc3MKICBHUFVDb21tYW5kRW5jb2Rlci5wcm90b3R5cGUuYmVnaW5Db21wdXRlUGFzcyA9IGZ1bmN0aW9uIChkZXNjKSB7CiAgICBpZiAoIVQuYXJtZWQgfHwgZW5jRGV2aWNlLmdldCh0aGlzKSAhPT0gZGV2aWNlIHx8IGRlc2M/LnRpbWVzdGFtcFdyaXRlcykgcmV0dXJuIG9yaWdCZWdpbi5jYWxsKHRoaXMsIGRlc2MpCiAgICBjb25zdCBhID0gYWxsb2NRdWVyeSgpCiAgICBpZiAoIWEpIHJldHVybiBvcmlnQmVnaW4uY2FsbCh0aGlzLCBkZXNjKQogICAgY29uc3QgcGFzcyA9IG9yaWdCZWdpbi5jYWxsKHRoaXMsIHdpdGhUcyhkZXNjLCBhKSkKICAgIHN0LnNldChwYXNzLCB7IGVuYzogdGhpcywgZGVzYywgY3VyOiBwYXNzLCByZWM6IG5ld1JlYyhhLCBULmNodW5rKSwgY2h1bms6IFQuY2h1bmssIHBpcGVsaW5lOiBudWxsLCBwaWQ6IC0xLCBiaW5kOiBuZXcgTWFwKCksIHVzZWQ6IGZhbHNlIH0pCiAgICByZXR1cm4gcGFzcwogIH0KCiAgY29uc3QgUCA9IEdQVUNvbXB1dGVQYXNzRW5jb2Rlci5wcm90b3R5cGUKICBjb25zdCBvcmlnU2V0UGlwZWxpbmUgPSBQLnNldFBpcGVsaW5lCiAgY29uc3Qgb3JpZ1NldEJpbmRHcm91cCA9IFAuc2V0QmluZEdyb3VwCiAgY29uc3Qgb3JpZ0Rpc3BhdGNoID0gUC5kaXNwYXRjaFdvcmtncm91cHMKICBjb25zdCBvcmlnRW5kID0gUC5lbmQKCiAgUC5zZXRQaXBlbGluZSA9IGZ1bmN0aW9uIChwKSB7CiAgICBjb25zdCBzID0gc3QuZ2V0KHRoaXMpCiAgICBpZiAoIXMpIHJldHVybiBvcmlnU2V0UGlwZWxpbmUuY2FsbCh0aGlzLCBwKQogICAgcy5waXBlbGluZSA9IHAKICAgIHMucGlkID0gcGlwZWxpbmVJZHMuZ2V0KHApID8/IC0xCiAgICByZXR1cm4gb3JpZ1NldFBpcGVsaW5lLmNhbGwocy5jdXIsIHApCiAgfQogIFAuc2V0QmluZEdyb3VwID0gZnVuY3Rpb24gKC4uLmFyZ3MpIHsKICAgIGNvbnN0IHMgPSBzdC5nZXQodGhpcykKICAgIGlmICghcykgcmV0dXJuIG9yaWdTZXRCaW5kR3JvdXAuYXBwbHkodGhpcywgYXJncykKICAgIHMuYmluZC5zZXQoYXJnc1swXSwgYXJncykKICAgIHJldHVybiBvcmlnU2V0QmluZEdyb3VwLmFwcGx5KHMuY3VyLCBhcmdzKQogIH0KICBQLmRpc3BhdGNoV29ya2dyb3VwcyA9IGZ1bmN0aW9uICguLi5hcmdzKSB7CiAgICBjb25zdCBzID0gc3QuZ2V0KHRoaXMpCiAgICBpZiAoIXMpIHJldHVybiBvcmlnRGlzcGF0Y2guYXBwbHkodGhpcywgYXJncykKICAgIGlmIChTUExJVCAmJiBzLnVzZWQpIHsKICAgICAgY29uc3QgYSA9IGFsbG9jUXVlcnkoKQogICAgICBpZiAoYSkgewogICAgICAgIG9yaWdFbmQuY2FsbChzLmN1cikKICAgICAgICBzLmN1ciA9IG9yaWdCZWdpbi5jYWxsKHMuZW5jLCB3aXRoVHMocy5kZXNjLCBhKSkKICAgICAgICBzLnJlYyA9IG5ld1JlYyhhLCBzLmNodW5rKQogICAgICAgIGlmIChzLnBpcGVsaW5lKSBvcmlnU2V0UGlwZWxpbmUuY2FsbChzLmN1ciwgcy5waXBlbGluZSkKICAgICAgICBmb3IgKGNvbnN0IGJnIG9mIHMuYmluZC52YWx1ZXMoKSkgb3JpZ1NldEJpbmRHcm91cC5hcHBseShzLmN1ciwgYmcpCiAgICAgIH0KICAgIH0KICAgIHMudXNlZCA9IHRydWUKICAgIHMucmVjLmRpc3BhdGNoZXMrKwogICAgcy5yZWMucGlkID0gcy5waWQKICAgIHMucmVjLndnID0gYXJncy5zbGljZSgwLCAzKQogICAgcmV0dXJuIG9yaWdEaXNwYXRjaC5hcHBseShzLmN1ciwgYXJncykKICB9CiAgUC5lbmQgPSBmdW5jdGlvbiAoKSB7CiAgICBjb25zdCBzID0gc3QuZ2V0KHRoaXMpCiAgICByZXR1cm4gb3JpZ0VuZC5jYWxsKHMgPyBzLmN1ciA6IHRoaXMpCiAgfQogIGZvciAoY29uc3QgbmFtZSBvZiBbJ3B1c2hEZWJ1Z0dyb3VwJywgJ3BvcERlYnVnR3JvdXAnLCAnaW5zZXJ0RGVidWdNYXJrZXInXSkgewogICAgY29uc3Qgb3JpZyA9IFBbbmFtZV0KICAgIFBbbmFtZV0gPSBmdW5jdGlvbiAoLi4uYXJncykgewogICAgICBjb25zdCBzID0gc3QuZ2V0KHRoaXMpCiAgICAgIGlmIChzKSBULmRlYnVnR3JvdXBDYWxscysrCiAgICAgIHJldHVybiBvcmlnLmFwcGx5KHMgPyBzLmN1ciA6IHRoaXMsIGFyZ3MpCiAgICB9CiAgfQoKICAvLyBSZXNvbHZlIHdpdGggdGhlIHVud3JhcHBlZCBtZXRob2RzIHNvIHRoaXMgcmVhZGJhY2sgaXMgbm90IGl0c2VsZiByZWNvcmRlZC4KICB3aW5kb3cuX19yZWFkR3B1ID0gYXN5bmMgKCkgPT4gewogICAgaWYgKCFkZXZpY2UpIHsgVC5ncHVOb3RlID0gJ3RpbWVzdGFtcC1xdWVyeSB1bmF2YWlsYWJsZSBvbiB0aGlzIGRldmljZSc7IHJldHVybiB9CiAgICBpZiAobmV4dFF1ZXJ5ID09PSAwKSB7IFQuZ3B1Tm90ZSA9ICdubyBwYXNzZXMgd2VyZSBpbnN0cnVtZW50ZWQnOyByZXR1cm4gfQogICAgY29uc3QgZW5jID0gb3JpZ0NyZWF0ZUVuYy5jYWxsKGRldmljZSkKICAgIGNvbnN0IHJlYWRzID0gc2V0cy5tYXAoKHFzLCBzaSkgPT4gewogICAgICBjb25zdCBuID0gTWF0aC5taW4oTUFYUSwgbmV4dFF1ZXJ5IC0gc2kgKiBNQVhRKQogICAgICBjb25zdCByZXNvbHZlQnVmID0gZGV2aWNlLmNyZWF0ZUJ1ZmZlcih7IHNpemU6IG4gKiA4LCB1c2FnZTogR1BVQnVmZmVyVXNhZ2UuUVVFUllfUkVTT0xWRSB8IEdQVUJ1ZmZlclVzYWdlLkNPUFlfU1JDIH0pCiAgICAgIGNvbnN0IHJlYWRCdWYgPSBkZXZpY2UuY3JlYXRlQnVmZmVyKHsgc2l6ZTogbiAqIDgsIHVzYWdlOiBHUFVCdWZmZXJVc2FnZS5DT1BZX0RTVCB8IEdQVUJ1ZmZlclVzYWdlLk1BUF9SRUFEIH0pCiAgICAgIGVuYy5yZXNvbHZlUXVlcnlTZXQocXMsIDAsIG4sIHJlc29sdmVCdWYsIDApCiAgICAgIGVuYy5jb3B5QnVmZmVyVG9CdWZmZXIocmVzb2x2ZUJ1ZiwgMCwgcmVhZEJ1ZiwgMCwgbiAqIDgpCiAgICAgIHJldHVybiByZWFkQnVmCiAgICB9KQogICAgZGV2aWNlLnF1ZXVlLnN1Ym1pdChbZW5jLmZpbmlzaCgpXSkKICAgIGNvbnN0IHRzID0gW10KICAgIGZvciAoY29uc3QgcmVhZEJ1ZiBvZiByZWFkcykgewogICAgICBhd2FpdCBvcmlnTWFwQXN5bmMuY2FsbChyZWFkQnVmLCBHUFVNYXBNb2RlLlJFQUQpCiAgICAgIHRzLnB1c2gobmV3IEJpZ1VpbnQ2NEFycmF5KHJlYWRCdWYuZ2V0TWFwcGVkUmFuZ2UoKS5zbGljZSgwKSkpCiAgICAgIHJlYWRCdWYudW5tYXAoKQogICAgfQogICAgbGV0IGJhc2UgPSBudWxsCiAgICBmb3IgKGNvbnN0IGFyciBvZiB0cykgZm9yIChjb25zdCB2IG9mIGFycikgaWYgKHYgIT09IDBuICYmIChiYXNlID09PSBudWxsIHx8IHYgPCBiYXNlKSkgYmFzZSA9IHYKICAgIGZvciAoY29uc3QgcCBvZiBULnBhc3NlcykgewogICAgICBwLmJlZ2luTnMgPSBOdW1iZXIodHNbcC5zaV1bcC5xXSAtIGJhc2UpCiAgICAgIHAuZW5kTnMgPSBOdW1iZXIodHNbcC5zaV1bcC5xICsgMV0gLSBiYXNlKQogICAgfQogICAgVC5ncHVOb3RlID0gYCR7VC5wYXNzZXMubGVuZ3RofSBwYXNzZXMgaW5zdHJ1bWVudGVkIGluICR7c2V0cy5sZW5ndGh9IHF1ZXJ5IHNldChzKWAgKwogICAgICAobmV4dFF1ZXJ5ID49IE1BWFEgKiBNQVhfU0VUUyA/ICcgKHF1ZXJ5IHNldHMgZnVsbCknIDogJycpCiAgfQp9KSgpCjwvc2NyaXB0PgoKPHNjcmlwdCB0eXBlPSJtb2R1bGUiPgogIGNvbnN0IGxvZyA9IChzKSA9PiB7IGRvY3VtZW50LmdldEVsZW1lbnRCeUlkKCdsb2cnKS5hcHBlbmQocyArICdcbicpOyBjb25zb2xlLmxvZygnW3RpbWluZ10gJyArIHMpIH0KICBjb25zdCBwYXJhbXMgPSBuZXcgVVJMU2VhcmNoUGFyYW1zKGxvY2F0aW9uLnNlYXJjaCkKICBjb25zdCBWRVJTSU9OID0gcGFyYW1zLmdldCgndicpID8/ICcwLjE3LjEnCiAgY29uc3QgTUFYX0NIVU5LUyA9IE51bWJlcihwYXJhbXMuZ2V0KCdjaHVua3MnKSA/PyAyMDApCiAgY29uc3QgVCA9IHdpbmRvdy5fX3RpbWluZwogIC8vIGdwdTogdGltZXN0YW1wIHRoZSB3aG9sZSB3YXJtIG1lc3NhZ2UgYWZ0ZXIgaXRzIGZpcnN0IHRva2Vucy4KICAvLyBrZXJuZWw6IHNwbGl0IG9ubHkgYSBzaG9ydCB3aW5kb3csIGl0IGlzIGV4cGVuc2l2ZS4KICBjb25zdCBBUk1fQVQgPSBULm1vZGUgPT09ICdrZXJuZWwnID8gMTAgOiA0CiAgY29uc3QgRElTQVJNX0FUID0gVC5tb2RlID09PSAna2VybmVsJyA/IDE3IDogSW5maW5pdHkKICB0cnkgewogICAgY29uc3QgYWRhcHRlciA9IGF3YWl0IG5hdmlnYXRvci5ncHU/LnJlcXVlc3RBZGFwdGVyKHsgcG93ZXJQcmVmZXJlbmNlOiAnaGlnaC1wZXJmb3JtYW5jZScgfSkKICAgIFQuYWRhcHRlciA9IGFkYXB0ZXIgPyB7IHZlbmRvcjogYWRhcHRlci5pbmZvPy52ZW5kb3IsIGFyY2hpdGVjdHVyZTogYWRhcHRlci5pbmZvPy5hcmNoaXRlY3R1cmUsIGRldmljZTogYWRhcHRlci5pbmZvPy5kZXZpY2UsIGRlc2NyaXB0aW9uOiBhZGFwdGVyLmluZm8/LmRlc2NyaXB0aW9uIH0gOiBudWxsCiAgICBULnZlcnNpb24gPSBWRVJTSU9OCiAgICBsb2coYG1vZGU9JHtULm1vZGV9IGFkYXB0ZXI9JHtKU09OLnN0cmluZ2lmeShULmFkYXB0ZXIpfSBpc29sYXRlZD0ke3NlbGYuY3Jvc3NPcmlnaW5Jc29sYXRlZH1gKQoKICAgIGNvbnN0IHsgRW5naW5lIH0gPSBhd2FpdCBpbXBvcnQoYGh0dHBzOi8vY2RuLmpzZGVsaXZyLm5ldC9ucG0vQGxpdGVydC1sbS9jb3JlQCR7VkVSU0lPTn0vK2VzbWApCiAgICBfX21hcmsoJ2ltcG9ydC1kb25lJykKICAgIGNvbnN0IGVuZ2luZSA9IGF3YWl0IEVuZ2luZS5jcmVhdGUoeyBtb2RlbDogJy9nZW1tYS00LUUyQi1pdC13ZWIubGl0ZXJ0bG0nIH0pCiAgICBfX21hcmsoJ2VuZ2luZS1jcmVhdGVkJykKCiAgICBjb25zdCBydW4gPSBhc3luYyAobGFiZWwsIHByb21wdCwgYXJtKSA9PiB7CiAgICAgIGNvbnN0IGNoYXQgPSBhd2FpdCBlbmdpbmUuY3JlYXRlQ29udmVyc2F0aW9uKCkKICAgICAgbGV0IHRleHQgPSAnJwogICAgICBsZXQgbiA9IDAKICAgICAgX19tYXJrKGAke2xhYmVsfTpzZW5kYCkKICAgICAgZm9yIGF3YWl0IChjb25zdCBjaHVuayBvZiBjaGF0LnNlbmRNZXNzYWdlU3RyZWFtaW5nKHByb21wdCkpIHsKICAgICAgICB0ZXh0ICs9IGNodW5rPy5jb250ZW50Py5bMF0/LnRleHQgPz8gJycKICAgICAgICBULmNodW5rKysKICAgICAgICBfX21hcmsoYCR7bGFiZWx9OmNodW5rLSR7bn1gKQogICAgICAgIGlmIChhcm0gJiYgbiA9PT0gQVJNX0FUKSBULmFybWVkID0gdHJ1ZQogICAgICAgIGlmIChhcm0gJiYgbiA9PT0gRElTQVJNX0FUKSBULmFybWVkID0gZmFsc2UKICAgICAgICBpZiAoKytuID49IE1BWF9DSFVOS1MpIGJyZWFrCiAgICAgIH0KICAgICAgVC5hcm1lZCA9IGZhbHNlCiAgICAgIF9fbWFyayhgJHtsYWJlbH06ZG9uZWApCiAgICAgIGF3YWl0IGNoYXQuZGVsZXRlKCkKICAgICAgbG9nKGAke2xhYmVsfTogJHtufSBjaHVua3M6ICR7dGV4dC5zbGljZSgwLCAxNjApfWApCiAgICAgIHJldHVybiB7IHRleHQsIGNodW5rczogbiB9CiAgICB9CgogICAgLy8gRGlmZmVyZW50IHByb21wdHMgc28gdGhlIHdhcm0gcnVuIGNhbm5vdCBiZSBzZXJ2ZWQgZnJvbSBhIHNoYXJlZCBwcmVmaXguCiAgICBULmNvbGQgPSBhd2FpdCBydW4oJ2NvbGQnLCAnQ291bnQgZnJvbSBvbmUgdG8gdGhpcnR5IGluIEVuZ2xpc2ggd29yZHMsIHNlcGFyYXRlZCBieSBjb21tYXMuJywgZmFsc2UpCiAgICBULndhcm0gPSBhd2FpdCBydW4oJ3dhcm0nLCAnQ291bnQgZG93biBmcm9tIHRoaXJ0eSB0byBvbmUgaW4gRW5nbGlzaCB3b3Jkcywgc2VwYXJhdGVkIGJ5IGNvbW1hcy4nLCB0cnVlKQogICAgaWYgKHdpbmRvdy5fX3JlYWRHcHUpIGF3YWl0IHdpbmRvdy5fX3JlYWRHcHUoKQogICAgbG9nKGBncHU6ICR7VC5ncHVOb3RlIHx8ICduL2EnfTsgdmFsaWRhdGlvbiBlcnJvcnM6ICR7VC5ncHVFcnJvcnMubGVuZ3RofTsgZGVidWctZ3JvdXAgY2FsbHM6ICR7VC5kZWJ1Z0dyb3VwQ2FsbHN9YCkKICB9IGNhdGNoIChlKSB7CiAgICBULmVycm9yID0gU3RyaW5nKGU/LnN0YWNrID8/IGUpCiAgICBsb2coJ0VSUk9SOiAnICsgVC5lcnJvcikKICB9CiAgYXdhaXQgd2luZG93Ll9fc2V0dGxlSGFzaGVzPy4oKSAvLyBhYnNlbnQgaW4gbW9kZT1iYXJlLCB3aGljaCBob29rcyBub3RoaW5nIGJ1dCBtYXBBc3luYwogIGlmIChULnNoYWRlclNoYTI1NlVuYXZhaWxhYmxlKSBsb2coJ1dBUk5JTkc6IGNyeXB0by5zdWJ0bGUgdW5hdmFpbGFibGUsIHNoYWRlciBoYXNoZXMgbm90IHJlY29yZGVkJykKICB3aW5kb3cuX190aW1pbmdEb25lID0gdHJ1ZQo8L3NjcmlwdD4K","run.mjs":"aW1wb3J0IHsgY3JlYXRlU2VydmVyIH0gZnJvbSAnbm9kZTpodHRwJwppbXBvcnQgeyBjcmVhdGVSZWFkU3RyZWFtLCBzdGF0U3luYywgd3JpdGVGaWxlU3luYyB9IGZyb20gJ25vZGU6ZnMnCmltcG9ydCB7IGpvaW4sIG5vcm1hbGl6ZSwgZXh0bmFtZSB9IGZyb20gJ25vZGU6cGF0aCcKaW1wb3J0IHB1cHBldGVlciBmcm9tICdwdXBwZXRlZXInCi8vIFNlcnZlcyB0aGlzIGRpcmVjdG9yeSAodGltaW5nLmh0bWwgKyB0aGUgcGlubmVkIG1vZGVsIGJ1bmRsZSkgd2l0aAovLyBDT09QL0NPRVAsIHJ1bnMgd3d3L3RpbWluZy5odG1sIGJhcmUgdGhyb3VnaCBoZWFkbGVzcyBDaHJvbWUgb24gdGhlIFQ0LAovLyBhbmQgd3JpdGVzIHRoZSB2ZW5kb3ItbWF0cml4IHJvdyB0byB2ZW5kb3ItdDQtcm93Lmpzb24uCmNvbnN0IFJPT1QgPSBwcm9jZXNzLmN3ZCgpCmNvbnN0IE1JTUUgPSB7ICcuaHRtbCc6ICd0ZXh0L2h0bWwnLCAnLmpzJzogJ3RleHQvamF2YXNjcmlwdCcsICcud2FzbSc6ICdhcHBsaWNhdGlvbi93YXNtJyB9CmNvbnN0IHNlcnZlciA9IGNyZWF0ZVNlcnZlcigocmVxLCByZXMpID0+IHsKICBjb25zdCBmID0gam9pbihST09ULCBub3JtYWxpemUoZGVjb2RlVVJJQ29tcG9uZW50KG5ldyBVUkwocmVxLnVybCwgJ2h0dHA6Ly94JykucGF0aG5hbWUpKSkKICBsZXQgczsgdHJ5IHsgcyA9IHN0YXRTeW5jKGYpLnNpemUgfSBjYXRjaCB7IHJldHVybiByZXMud3JpdGVIZWFkKDQwNCkuZW5kKCkgfQogIHJlcy53cml0ZUhlYWQoMjAwLCB7ICdDb250ZW50LVR5cGUnOiBNSU1FW2V4dG5hbWUoZildID8/ICdhcHBsaWNhdGlvbi9vY3RldC1zdHJlYW0nLCAnQ29udGVudC1MZW5ndGgnOiBzLCAnQ3Jvc3MtT3JpZ2luLU9wZW5lci1Qb2xpY3knOiAnc2FtZS1vcmlnaW4nLCAnQ3Jvc3MtT3JpZ2luLUVtYmVkZGVyLVBvbGljeSc6ICdyZXF1aXJlLWNvcnAnIH0pCiAgY3JlYXRlUmVhZFN0cmVhbShmKS5waXBlKHJlcykKfSkKYXdhaXQgbmV3IFByb21pc2UoKHIpID0+IHNlcnZlci5saXN0ZW4oODkxNywgJzEyNy4wLjAuMScsIHIpKQpjb25zdCBicm93c2VyID0gYXdhaXQgcHVwcGV0ZWVyLmxhdW5jaCh7IGhlYWRsZXNzOiAnbmV3JywgYXJnczogWwogICctLW5vLXNhbmRib3gnLCAnLS11c2UtYW5nbGU9dnVsa2FuJywgJy0tZW5hYmxlLWZlYXR1cmVzPVZ1bGthbicsICctLWRpc2FibGUtdnVsa2FuLXN1cmZhY2UnLAogICctLWVuYWJsZS11bnNhZmUtd2ViZ3B1JywgJy0tZW5hYmxlLWRhd24tZmVhdHVyZXM9YWxsb3dfdW5zYWZlX2FwaXMsZGlzYWJsZV9hZGFwdGVyX2Jsb2NrbGlzdCcsCiAgJy0tZGlzYWJsZS1kYXduLWZlYXR1cmVzPXRpbWVzdGFtcF9xdWFudGl6YXRpb24nLCAnLS1pZ25vcmUtZ3B1LWJsb2NrbGlzdCcgXSwgcHJvdG9jb2xUaW1lb3V0OiAxMjAwMDAwIH0pCmNvbnN0IHBhZ2UgPSBhd2FpdCBicm93c2VyLm5ld1BhZ2UoKQpwYWdlLm9uKCdwYWdlZXJyb3InLCAoZSkgPT4gY29uc29sZS5sb2coJ1twYWdlZXJyb3JdJywgZS5tZXNzYWdlKSkKYXdhaXQgcGFnZS5nb3RvKCdodHRwOi8vMTI3LjAuMC4xOjg5MTcvdGltaW5nLmh0bWw/bW9kZT1iYXJlJmNodW5rcz0xNicsIHsgd2FpdFVudGlsOiAnbG9hZCcgfSkKYXdhaXQgcGFnZS53YWl0Rm9yRnVuY3Rpb24oJ3dpbmRvdy5fX3RpbWluZ0RvbmUgPT09IHRydWUnLCB7IHRpbWVvdXQ6IDEyMDAwMDAsIHBvbGxpbmc6IDEwMDAgfSkKY29uc3QgVCA9IGF3YWl0IHBhZ2UuZXZhbHVhdGUoKCkgPT4gd2luZG93Ll9fdGltaW5nKQpjb25zdCBncHVJbmZvID0gYXdhaXQgcGFnZS5ldmFsdWF0ZShhc3luYyAoKSA9PiB7IGNvbnN0IGEgPSBhd2FpdCBuYXZpZ2F0b3IuZ3B1LnJlcXVlc3RBZGFwdGVyKCk7IHJldHVybiB7IHZlbmRvcjogYT8uaW5mbz8udmVuZG9yLCBhcmNoaXRlY3R1cmU6IGE/LmluZm8/LmFyY2hpdGVjdHVyZSwgZGV2aWNlOiBhPy5pbmZvPy5kZXZpY2UsIGRlc2NyaXB0aW9uOiBhPy5pbmZvPy5kZXNjcmlwdGlvbiwgZmVhdHVyZXM6IFsuLi4oYT8uZmVhdHVyZXMgPz8gW10pXSB9IH0pCmNvbnN0IGNodW5rVCA9IFQubWFya3MuZmlsdGVyKChtKSA9PiBtLm5hbWUuc3RhcnRzV2l0aCgnd2FybTpjaHVuay0nKSkubWFwKChtKSA9PiBtLnQpCmNvbnN0IGdhcHMgPSBjaHVua1Quc2xpY2UoOCkubWFwKCh0LCBpKSA9PiB0IC0gY2h1bmtUWzggKyBpIC0gMV0pLnNsaWNlKDEpLnNvcnQoKHgsIHkpID0+IHggLSB5KQpjb25zdCBtZWQgPSBnYXBzLmxlbmd0aCA/IGdhcHNbTWF0aC5mbG9vcihnYXBzLmxlbmd0aCAvIDIpXSA6IE5hTgpjb25zdCByb3cgPSB7IHN0YXR1czogJ21lYXN1cmVkJywgdmVuZG9yOiBncHVJbmZvLnZlbmRvciwgYXJjaGl0ZWN0dXJlOiBncHVJbmZvLmFyY2hpdGVjdHVyZSwgZGV2aWNlTGFiZWw6ICdDb2xhYiBUNCAodGhpcyBydW50aW1lKScsIGJyb3dzZXI6IGF3YWl0IGJyb3dzZXIudmVyc2lvbigpLCBzaGFkZXJGMTY6IGdwdUluZm8uZmVhdHVyZXMuaW5jbHVkZXMoJ3NoYWRlci1mMTYnKSwgdGltZXN0YW1wUXVlcnk6IGdwdUluZm8uZmVhdHVyZXMuaW5jbHVkZXMoJ3RpbWVzdGFtcC1xdWVyeScpLCBwYWNrYWdlOiBULnZlcnNpb24sIG1vZGVsU2hhMjU2OiAnM2EwOGU4ZDk0ZTIzYjgxNGFlNTQxNDQ2OWMzNzBjNTAzODEzOTQ5YWNiOGNlYWExN2U0ZWJmOGEzNWFmMzViNScsIHN0ZWFkeVRva1BlclNlYzogMTAwMCAvIG1lZCwgc3RlYWR5TXNQZXJUb2tlbjogbWVkLCB3YXJtQ2h1bmtzOiBjaHVua1QubGVuZ3RoLCBlcnJvcjogVC5lcnJvciA/PyBudWxsLCBncHVJbmZvIH0KY29uc29sZS5sb2coJ0FEQVBURVInLCBKU09OLnN0cmluZ2lmeShncHVJbmZvKSkKaWYgKFQuZXJyb3IpIGNvbnNvbGUubG9nKCdQQUdFIEVSUk9SJywgVC5lcnJvcikKY29uc29sZS5sb2coJ3N0ZWFkeSB0b2svczonLCByb3cuc3RlYWR5VG9rUGVyU2VjKQp3cml0ZUZpbGVTeW5jKCd2ZW5kb3ItdDQtcm93Lmpzb24nLCBKU09OLnN0cmluZ2lmeShyb3csIG51bGwsIDIpKQpjb25zb2xlLmxvZygnc2F2ZWQgdmVuZG9yLXQ0LXJvdy5qc29uOyBjb3B5IGl0IGJhY2sgaW50byBvdXQvdmVuZG9yLW1hdHJpeC5qc29uIHJvd3MudDRfdGVzbGFfY29sYWInKQphd2FpdCBicm93c2VyLmNsb3NlKCk7IHNlcnZlci5jbG9zZSgpCg=="}
os.makedirs('vendor', exist_ok=True)
for name, data in files.items():
    with open(os.path.join('vendor', name), 'wb') as f:
        f.write(base64.b64decode(data))
!cd vendor && npm init -y > /dev/null && npm i --silent puppeteer@24.40.0 > /dev/null && echo installed
!curl -L -C - -o vendor/gemma-4-E2B-it-web.litertlm https://huggingface.co/litert-community/gemma-4-E2B-it-litert-lm/resolve/b3ca0d2f076785a8f4b2219ddbd2bdb99954eae1/gemma-4-E2B-it-web.litertlm
h = hashlib.sha256()
with open('vendor/gemma-4-E2B-it-web.litertlm', 'rb') as f:
    [h.update(c) for c in iter(lambda: f.read(1 << 26), b'')]
print('model sha256:', h.hexdigest())
assert h.hexdigest() == '3a08e8d94e23b814ae5414469c370c503813949acb8ceaa17e4ebf8a35af35b5', 'model bundle mismatch, stop here'

In [ ]:
# 3) Run. Prints the adapter, steady tok/s and the row file (2 GB model load + 16+16 chunks on T4).
!cd vendor && node run.mjs
print(open('vendor/vendor-t4-row.json').read()[:800])

## Copy-back instructions

1. Download `vendor/vendor-t4-row.json` from this runtime.
2. In the litert-capture repo, on a branch from `main`: merge the row into `out/vendor-matrix.json` under `rows.t4_tesla_colab` (flip `status` to `measured`, keep the parked Intel/AMD/Adreno rows), add this run's `manifest_sha256` join (see `shader-sha.mjs`).
3. Run the standard gates (`pnpm exec biome check .`, `node --check`, `node analyze.mjs 400`, `gitleaks`), commit, and merge — that fills the second vendor-matrix row.
4. If the driver prints `shader-f16 present: true` on the T4, keep the row but flag it: the T4 took the f16 path and is not the expected no-f16 contrast.